# Currency Exchange Rate Prediction Analysis

This notebook demonstrates how to use the currency_predictor package for analyzing and predicting currency exchange rates.

In [ ]:
# Import required libraries
import sys
import os
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from currency_predictor import (
    CurrencyDataCollector,
    DataProcessor,
    CurrencyPredictor,
    setup_logging
)

# Setup
setup_logging(level="INFO")
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

## 1. Data Collection

Let's start by collecting some currency data from Yahoo Finance.

In [ ]:
# Initialize data collector
collector = CurrencyDataCollector()

# Collect data for EUR/USD
currency_pair = "EURUSD=X"
print(f"Collecting data for {currency_pair}...")

data = collector.get_yahoo_finance_data(currency_pair, period="2y")
print(f"Collected {len(data)} records")

# Display basic info
print("\nData shape:", data.shape)
print("\nColumns:", list(data.columns))
print("\nFirst few rows:")
data.head()

## 2. Data Visualization

Let's visualize the currency data to understand trends and patterns.

In [ ]:
# Plot price over time
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Price chart
axes[0].plot(data.index, data['Close'], label='Close Price', linewidth=1)
axes[0].set_title(f'{currency_pair} Exchange Rate Over Time', fontsize=14)
axes[0].set_ylabel('Exchange Rate')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Volume chart
axes[1].bar(data.index, data['Volume'], alpha=0.7, color='orange')
axes[1].set_title('Trading Volume', fontsize=14)
axes[1].set_ylabel('Volume')
axes[1].set_xlabel('Date')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Data Processing

Now let's process the data by creating technical indicators and features.

In [ ]:
# Initialize data processor
processor = DataProcessor()

# Clean the data
clean_data = processor.clean_data(data)
print(f"Clean data shape: {clean_data.shape}")

# Create technical indicators
processed_data = processor.create_technical_indicators(clean_data)
print(f"Data with technical indicators: {processed_data.shape}")

# Display some of the new features
print("\nNew technical indicator columns:")
new_cols = [col for col in processed_data.columns if col not in clean_data.columns]
print(new_cols[:10])  # Show first 10 new columns

In [ ]:
# Visualize some technical indicators
fig, axes = plt.subplots(3, 1, figsize=(15, 12))

# Price with moving averages
axes[0].plot(processed_data.index, processed_data['Close'], label='Close', linewidth=1)
axes[0].plot(processed_data.index, processed_data['MA_20'], label='MA 20', alpha=0.7)
axes[0].plot(processed_data.index, processed_data['MA_50'], label='MA 50', alpha=0.7)
axes[0].set_title('Price with Moving Averages')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# RSI
axes[1].plot(processed_data.index, processed_data['RSI'], color='purple')
axes[1].axhline(y=70, color='r', linestyle='--', alpha=0.7, label='Overbought')
axes[1].axhline(y=30, color='g', linestyle='--', alpha=0.7, label='Oversold')
axes[1].set_title('Relative Strength Index (RSI)')
axes[1].set_ylabel('RSI')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# MACD
axes[2].plot(processed_data.index, processed_data['MACD'], label='MACD', linewidth=1)
axes[2].plot(processed_data.index, processed_data['MACD_Signal'], label='Signal', linewidth=1)
axes[2].bar(processed_data.index, processed_data['MACD_Histogram'], 
           alpha=0.3, color='gray', label='Histogram')
axes[2].set_title('MACD')
axes[2].set_xlabel('Date')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Feature Engineering

Let's create lagged features and prepare the data for machine learning.

In [ ]:
# Create lagged features
final_data = processor.create_lagged_features(processed_data)
print(f"Data with lagged features: {final_data.shape}")

# Prepare features and target
X, y = processor.prepare_features_target(final_data, prediction_horizon=1)
print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

print(f"\nFeature columns ({len(X.columns)}):")
for i, col in enumerate(X.columns):
    print(f"{i+1:2d}. {col}")
    if i >= 19:  # Show first 20 features
        print(f"    ... and {len(X.columns)-20} more")
        break

## 5. Model Training

Now let's train multiple machine learning models to predict currency prices.

In [ ]:
# Split data into train/test sets
X_train, X_test, y_train, y_test = processor.train_test_split(X, y, test_size=0.2)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# Scale features
X_train_scaled, X_test_scaled = processor.scale_features(X_train, X_test)

print("\nFeatures scaled successfully")

In [ ]:
# Initialize and setup models
predictor = CurrencyPredictor()
predictor.setup_default_models()

print(f"Available models: {list(predictor.models.keys())}")

# Train all models
print("\nTraining models...")
predictor.train_all_models(X_train_scaled, y_train)

print(f"Trained models: {list(predictor.trained_models.keys())}")

## 6. Model Evaluation

Let's evaluate the performance of our trained models.

In [ ]:
# Evaluate all models
results = predictor.evaluate_all_models(X_test_scaled, y_test)

# Create a results DataFrame for better visualization
results_df = pd.DataFrame(results).T
print("Model Performance Results:")
print("=" * 60)
print(results_df.round(4))

# Find best model
best_model = predictor.get_best_model(results, metric='rmse')
print(f"\nBest model (lowest RMSE): {best_model}")

In [ ]:
# Visualize model performance
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

metrics = ['rmse', 'mae', 'r2', 'mape']
metric_names = ['RMSE', 'MAE', 'R²', 'MAPE (%)']

for i, (metric, name) in enumerate(zip(metrics, metric_names)):
    ax = axes[i//2, i%2]
    values = [results[model][metric] for model in results.keys()]
    models = list(results.keys())
    
    bars = ax.bar(models, values, color='skyblue', alpha=0.7)
    ax.set_title(f'{name} by Model')
    ax.set_ylabel(name)
    
    # Highlight best model
    if metric in ['rmse', 'mae', 'mape']:
        best_idx = np.argmin(values)
    else:  # r2
        best_idx = np.argmax(values)
    
    bars[best_idx].set_color('orange')
    
    # Rotate x-axis labels
    ax.tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for bar, value in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{value:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

## 7. Feature Importance

Let's analyze which features are most important for our best performing model.

In [ ]:
# Get feature importance for the best model
importance = predictor.get_feature_importance(best_model, top_n=15)

if importance:
    # Create DataFrame for easier plotting
    importance_df = pd.DataFrame(list(importance.items()), 
                               columns=['Feature', 'Importance'])
    
    # Plot feature importance
    plt.figure(figsize=(12, 8))
    bars = plt.barh(importance_df['Feature'], importance_df['Importance'])
    plt.title(f'Top 15 Feature Importance - {best_model.title()}')
    plt.xlabel('Importance')
    
    # Color bars
    bars[0].set_color('orange')  # Highlight most important
    
    plt.tight_layout()
    plt.show()
    
    print("Top 10 Most Important Features:")
    print("-" * 40)
    for i, (feature, score) in enumerate(list(importance.items())[:10], 1):
        print(f"{i:2d}. {feature:<25} {score:.4f}")
else:
    print(f"Feature importance not available for {best_model}")

## 8. Prediction Visualization

Let's visualize how well our best model predicts the actual currency prices.

In [ ]:
# Make predictions with the best model
y_pred = predictor.predict(best_model, X_test_scaled)

# Create prediction vs actual plot
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Time series plot
test_dates = y_test.index
axes[0].plot(test_dates, y_test.values, label='Actual', alpha=0.7, linewidth=2)
axes[0].plot(test_dates, y_pred, label='Predicted', alpha=0.7, linewidth=2)
axes[0].set_title(f'Actual vs Predicted - {best_model.title()}')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Exchange Rate')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Scatter plot
axes[1].scatter(y_test.values, y_pred, alpha=0.6)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
            'r--', alpha=0.8, label='Perfect Prediction')
axes[1].set_xlabel('Actual Values')
axes[1].set_ylabel('Predicted Values')
axes[1].set_title('Prediction Accuracy Scatter Plot')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate and display error statistics
residuals = y_test.values - y_pred
print(f"\nPrediction Error Analysis ({best_model}):")
print("-" * 40)
print(f"Mean Residual: {np.mean(residuals):.6f}")
print(f"Std Residual:  {np.std(residuals):.6f}")
print(f"RMSE:          {results[best_model]['rmse']:.6f}")
print(f"R² Score:      {results[best_model]['r2']:.4f}")

## 9. Residual Analysis

Let's analyze the prediction residuals to understand model performance better.

In [ ]:
# Residual analysis
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Residuals over time
axes[0].plot(test_dates, residuals, alpha=0.7)
axes[0].axhline(y=0, color='r', linestyle='--', alpha=0.8)
axes[0].set_title('Residuals Over Time')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Residual')
axes[0].grid(True, alpha=0.3)

# Residual distribution
axes[1].hist(residuals, bins=30, alpha=0.7, density=True)
axes[1].set_title('Residual Distribution')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Density')
axes[1].grid(True, alpha=0.3)

# Q-Q plot for normality check
from scipy import stats
stats.probplot(residuals, dist="norm", plot=axes[2])
axes[2].set_title('Q-Q Plot (Normality Check)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistical tests
from scipy.stats import shapiro, jarque_bera

# Normality test
shapiro_stat, shapiro_p = shapiro(residuals)
jb_stat, jb_p = jarque_bera(residuals)

print("\nResidual Analysis:")
print("-" * 30)
print(f"Shapiro-Wilk Test (normality):")
print(f"  Statistic: {shapiro_stat:.4f}")
print(f"  p-value:   {shapiro_p:.4f}")
print(f"\nJarque-Bera Test (normality):")
print(f"  Statistic: {jb_stat:.4f}")
print(f"  p-value:   {jb_p:.4f}")

if shapiro_p > 0.05:
    print("\n✓ Residuals appear to be normally distributed (good!)")
else:
    print("\n⚠ Residuals may not be normally distributed")

## 10. Conclusion

Summary of our currency prediction analysis.

In [ ]:
print("\n" + "="*60)
print("CURRENCY PREDICTION ANALYSIS SUMMARY")
print("="*60)

print(f"\n📊 Dataset: {currency_pair}")
print(f"📅 Period: {data.index[0].strftime('%Y-%m-%d')} to {data.index[-1].strftime('%Y-%m-%d')}")
print(f"📈 Total Records: {len(data):,}")
print(f"🔧 Features Created: {X.shape[1]}")
print(f"🎯 Training Samples: {len(X_train):,}")
print(f"🧪 Test Samples: {len(X_test):,}")

print(f"\n🏆 Best Model: {best_model.upper()}")
print(f"📏 RMSE: {results[best_model]['rmse']:.6f}")
print(f"📐 MAE: {results[best_model]['mae']:.6f}")
print(f"📊 R² Score: {results[best_model]['r2']:.4f}")
print(f"📋 MAPE: {results[best_model]['mape']:.2f}%")

if importance:
    top_feature = list(importance.keys())[0]
    print(f"\n⭐ Most Important Feature: {top_feature}")

print("\n💡 Key Insights:")
r2_score = results[best_model]['r2']
if r2_score > 0.8:
    print("   - Excellent model performance with high predictive power")
elif r2_score > 0.6:
    print("   - Good model performance with reasonable predictive power")
elif r2_score > 0.4:
    print("   - Moderate model performance, room for improvement")
else:
    print("   - Model performance could be improved significantly")

mape = results[best_model]['mape']
if mape < 1:
    print("   - Very low prediction error - excellent accuracy")
elif mape < 3:
    print("   - Low prediction error - good accuracy")
elif mape < 5:
    print("   - Moderate prediction error - acceptable accuracy")
else:
    print("   - High prediction error - consider model improvements")

print("\n🔄 Next Steps:")
print("   - Experiment with different time horizons")
print("   - Try ensemble methods combining multiple models")
print("   - Include additional economic indicators")
print("   - Implement walk-forward validation")
print("   - Consider deep learning approaches (LSTM, GRU)")

print("\n" + "="*60)